# From Sight to Action — extracting a visual→motor pathway

**Question.** Which neurons form a structural route from the R1–R6
photoreceptors to **DNg13**, a descending neuron of the locomotor control
population, in the adult male *Drosophila* central nervous system?

**Data.** MaleCNS v1.0 (HHMI Janelia / Cambridge Drosophila Connectomics /
Google Research), CC-BY. Flat connectome tables are downloaded from the
public bucket `gs://flyem-male-cns/v1.0/connectome-data/flat-connectome/`
and neuron skeletons from
`gs://flyem-male-cns/v1.0/segmentation/skeletons-malecns/skeletons-swc/`.
No API token is required. See `../README.md` for the download commands.

**Scope.** Everything below is a *structural* analysis of wiring. Nothing
here simulates neural activity or predicts behaviour.

In [1]:
import sys

sys.path.insert(0, "../src")

import pandas as pd

from sight_to_action.data import load_annotations, load_neurotransmitters, load_weights
from sight_to_action.analysis import (
    build_type_graph,
    strongest_paths,
    bottleneck_nodes,
    removal_effect,
)

annotations = load_annotations()
neurotransmitters = load_neurotransmitters()
weights = load_weights()

print(f"{len(annotations):,} traced bodies")
print(f"{len(weights):,} body→body connections")

165,122 traced bodies
25,568,639 body→body connections


## 1. Confirm the endpoints exist

R1–R6 are the outer photoreceptors (the motion/luminance channel).
DNg13 is a descending neuron. Both must be present and traced before any
path search is meaningful.

In [2]:
for t in ["R1-R6", "DNg13"]:
    sub = annotations[annotations["type"] == t]
    print(
        f"{t:8s} n_bodies={len(sub):5d}  superclass={sub['superclass'].mode().iat[0]}"
        f"  dimorphism={sub['dimorphism'].dropna().unique()}"
    )

R1-R6    n_bodies= 1394  superclass=ol_sensory  dimorphism=<ArrowStringArray>
[]
Length: 0, dtype: str
DNg13    n_bodies=    2  superclass=descending_neuron  dimorphism=<ArrowStringArray>
['sexually dimorphic']
Length: 1, dtype: str


## 2. Build the type-level graph

Every body→body connection is summed into a type→type edge. Each edge
additionally gets a **relative weight**: the fraction of the *target*
type's total input that arrives from that source. Raw synapse counts alone
favour large, highly-connected cell types; the relative weight asks the
more useful question, "how much of what this neuron hears comes from
there?".

In [3]:
graph = build_type_graph(weights, min_weight=20)
print(f"{graph.number_of_nodes():,} cell types, {graph.number_of_edges():,} edges")

11,736 cell types, 583,744 edges


## 3. Rank the routes

A route's score is the **product of relative weights** along it, found as a
shortest path under the additive cost `-log(relative weight)`. Searching is
restricted to a corridor of types that can lie on a ≤5-hop path, which
makes the k-shortest-path search tractable without changing the result.

In [4]:
routes = strongest_paths(graph, "R1-R6", "DNg13", k=12, max_hops=5)

rows = []
for i, r in enumerate(routes, 1):
    rows.append(
        {
            "rank": i,
            "hops": r.hops,
            "route": " → ".join(r.nodes),
            "total_synapses": r.total_synapses,
            "weakest_link": r.min_relative_weight,
            "score": r.score,
        }
    )
pd.DataFrame(rows)

,rank,hops,route,total_synapses,weakest_link,score
0,1,4,R1-R6 → L3 → Tm5c → LoVP92 → DNg13,61273,0.00531,0.000004
1,2,5,R1-R6 → L1 → Tm3 → LC10a → AOTU002_b → DNg13,211589,0.00924,0.000002
2,3,5,R1-R6 → L1 → Tm3 → LC10a → AOTU002_c → DNg13,210693,0.00787,0.000002
3,4,5,R1-R6 → L1 → Tm3 → LC10a → AOTU016_c → DNg13,211427,0.00767,0.000002
4,5,4,R1-R6 → L1 → Tm3 → LoVP92 → DNg13,203591,0.00531,0.000002
5,6,5,R1-R6 → L1 → Tm3 → LC10a → AOTU002_a → DNg13,211116,0.00600,0.000002
6,7,4,R1-R6 → L2 → Tm4 → LoVP92 → DNg13,217724,0.00531,0.000002
7,8,5,R1-R6 → L3 → Tm20 → LC10c-1 → AOTU002_b → DNg13,84022,0.00924,0.000001
8,9,5,R1-R6 → L1 → Mi1 → Y3 → LoVP90b → DNg13,308780,0.00413,0.000001
9,10,5,R1-R6 → L1 → Tm3 → LC10d → AOTU002_b → DNg13,208411,0.00924,0.000001


The routes recapitulate the textbook fly visual pathway —
**retina → lamina → medulla → lobula → central brain → descending neuron** —
without that structure being imposed anywhere in the search. The lamina
monopolar cells (L1/L2/L3), medulla neurons (Tm/Mi), lobula projection
neurons (LC10 family, LoVP) and anterior-optic-tubercle cells (AOTU002/016)
all appear because of measured connectivity alone.

## 4. Bottlenecks and alternative routes

A cell type appearing in many of the alternative routes is a structural
bottleneck: if it were missing, those particular wiring routes would not
exist. This is a claim about graph connectivity, not about function.

In [5]:
bottlenecks = bottleneck_nodes(routes)
bottlenecks.head(10)

,type,n_routes,fraction_of_routes
0,L1,7,0.583333
1,Tm3,6,0.500000
2,LC10a,5,0.416667
3,LoVP92,4,0.333333
4,AOTU002_b,4,0.333333
5,L3,3,0.250000
6,Tm5c,2,0.166667
7,L2,2,0.166667
8,Tm4,2,0.166667
9,Mi1,1,0.083333


## 5. Node-removal experiment

Deleting a type from the graph and re-running the search shows how the
available structural routes change. Again: this is **not** a simulation of
activity and **not** a prediction of behaviour.

In [6]:
for t in ["LC10a", "LoVP92", "Tm3"]:
    eff = removal_effect(graph, "R1-R6", "DNg13", t, max_hops=5)
    print(f"remove {t}:")
    print(f"   before: {' → '.join(eff['best_before'])}")
    print(f"   after : {' → '.join(eff['best_after']) if eff['best_after'] else 'no route within hop limit'}")
    print(f"   still connected: {eff['still_connected']}\n")

remove LC10a:
   before: R1-R6 → L3 → Tm5c → LoVP92 → DNg13
   after : R1-R6 → L3 → Tm5c → LoVP92 → DNg13
   still connected: True



remove LoVP92:
   before: R1-R6 → L3 → Tm5c → LoVP92 → DNg13
   after : R1-R6 → L1 → Tm3 → LC10a → AOTU002_b → DNg13
   still connected: True



remove Tm3:
   before: R1-R6 → L3 → Tm5c → LoVP92 → DNg13
   after : R1-R6 → L3 → Tm5c → LoVP92 → DNg13
   still connected: True



## 6. Male / female comparison

MaleCNS annotations carry cross-connectome mappings (`flywireType` for the
female FlyWire/FAFB dataset, `hemibrainType` for the female hemibrain) and
explicit `dimorphism` flags. This lets each neuron on the pathway be
classified as shared, sex-specific or sexually dimorphic.

In [7]:
pathway_types = sorted({n for r in routes for n in r.nodes})
traced = annotations[annotations["status"] == "Traced"]
sex_rows = []
for t in pathway_types:
    s = traced[traced["type"] == t]
    sex_rows.append(
        {
            "type": t,
            "n_bodies": len(s),
            "flywireType": s["flywireType"].dropna().iloc[0] if s["flywireType"].notna().any() else None,
            "hemibrainType": s["hemibrainType"].dropna().iloc[0] if s["hemibrainType"].notna().any() else None,
            "dimorphism": s["dimorphism"].dropna().iloc[0] if s["dimorphism"].notna().any() else None,
        }
    )
sex_df = pd.DataFrame(sex_rows)
sex_df

,type,n_bodies,flywireType,hemibrainType,dimorphism
0,AOTU002_a,5,CB2070,AOTU002_a,NaN
1,AOTU002_b,6,CB4184,AOTU002_b,NaN
2,AOTU002_c,4,CB1080,AOTU002_b,NaN
3,AOTU016_c,4,"CB0007,CB0739",AOTU016,NaN
4,DNg13,2,DNg13,hb1696530677,sexually dimorphic
5,L1,1776,L1,NaN,NaN
6,L2,1779,L2,NaN,NaN
7,L3,1772,L3,NaN,NaN
8,LC10a,275,LC10a,NaN,NaN
9,LC10c-1,130,LC10c,NaN,NaN


**Finding.** The pathway is not sex-neutral. `LoVP92` — which sits on the
top-ranked route — is annotated **male-specific**, `VES200m` is
*potentially* male-specific, and the target neuron **DNg13 is annotated
sexually dimorphic** with a named FlyWire counterpart. So the strongest
structural route found here may have no direct equivalent in the female
connectome, while the early visual stages (photoreceptors, lamina, medulla)
map cleanly onto female cell types.

In [8]:
sex_df[sex_df["dimorphism"].notna()]

,type,n_bodies,flywireType,hemibrainType,dimorphism
4,DNg13,2,DNg13,hb1696530677,sexually dimorphic
12,LoVP92,13,NaN,NaN,male-specific
19,VES200m,12,NaN,NaN,potentially male-specific


## 7. Is the route actually special?

This is the part that distinguishes the project from a path-finder. A dense
recurrent network connects nearly everything to nearly everything within a
few hops, so *a path existing* is weak evidence. Three attempts to falsify
the route follow.

In [9]:
from sight_to_action.nulls import best_scores_from, best_scores_to, evaluate

nulls = evaluate(graph, annotations, "R1-R6", "DNg13", max_hops=5, n_shuffles=50)
print(f"observed score            {nulls.observed_score:.3e}")
print(f"rank among descending     {nulls.target_rank} of {nulls.target_pool}")
print(f"rank among sensory types  {nulls.source_rank} of {nulls.source_pool}")
print(f"weight-shuffled mean      {nulls.shuffled_mean:.3e}")
print(f"shuffles >= observed      {nulls.shuffled_better_fraction:.0%} of {nulls.n_shuffles}")

observed score            3.953e-06
rank among descending     312 of 480
rank among sensory types  191 of 333
weight-shuffled mean      1.492e-05
shuffles >= observed      70% of 50


### Which descending neurons *are* strongly wired from the photoreceptors?

This doubles as a sanity check on the method: if it is measuring anything
real, the descending neurons already known to be visually driven should come
out on top.

In [10]:
traced_all = annotations[annotations["status"] == "Traced"]
dn_types = set(traced_all[traced_all["superclass"] == "descending_neuron"]["type"].dropna())
forward = best_scores_from(graph, "R1-R6", max_hops=5)
top_dn = sorted(
    ((t, s) for t, s in forward.items() if t in dn_types and s > 0),
    key=lambda kv: kv[1],
    reverse=True,
)[:10]
pd.DataFrame(top_dn, columns=["descending_neuron", "score"])

,descending_neuron,score
0,DNc01,0.005755
1,DNc02,0.002839
2,DNp11,0.002294
3,DNp04,0.001808
4,DNp02,0.000897
5,DNg46,0.000895
6,DNp01,0.000555
7,DNg41,0.000504
8,DNge097,0.000477
9,DNp03,0.000445


`DNp01` — the Giant Fiber, the textbook visual escape neuron — appears near
the top at roughly 140× the score of DNg13, alongside DNp02/03/04 and DNp11.
The method is finding the visually-driven descending neurons; DNg13 simply
is not one of the strongest of them by this structural measure.

### Which senses dominate DNg13's input?

In [11]:
sensory_types = set(
    traced_all[traced_all["superclass"].isin(["ol_sensory", "cb_sensory", "vnc_sensory"])][
        "type"
    ].dropna()
)
backward = best_scores_to(graph, "DNg13", max_hops=5)
top_sensory = sorted(
    ((t, s) for t, s in backward.items() if t in sensory_types and s > 0),
    key=lambda kv: kv[1],
    reverse=True,
)[:10]
pd.DataFrame(top_sensory, columns=["sensory_type", "score"])

,sensory_type,score
0,SNpp10,0.001431
1,BM_Taste,0.000532
2,SNpp21,0.000516
3,SNpp50,0.000437
4,ORN_DA1,0.000242
5,SNpp48,0.000190
6,JO-A2,0.000171
7,SNta21,0.000125
8,SNppxx,0.000124
9,SNta38,0.000120


**Conclusion.** The R1–R6 → DNg13 route is real, reproducible and stable,
but it is *not statistically special*: below median against other descending
neurons, below median against other senses, and beaten by most
weight-shuffled graphs. That does not contradict the published result that
DNg13 is visually driven and steers walking — it shows that **path
existence, and even "strongest path", are weak evidence of a functional
channel**, which is precisely the failure mode invited by tools that return
a path between any two neurons.

### Robustness to analysis choices

In [12]:
rows = []
for min_weight in [10, 20, 50, 100]:
    g = build_type_graph(weights, min_weight=min_weight)
    found = strongest_paths(g, "R1-R6", "DNg13", k=1, max_hops=5)
    rows.append(
        {
            "min_weight": min_weight,
            "score": found[0].score if found else 0.0,
            "route": " → ".join(found[0].nodes) if found else "none",
        }
    )
pd.DataFrame(rows)

,min_weight,score,route
0,10,0.000004,R1-R6 → L3 → Tm5c → LoVP92 → DNg13
1,20,0.000004,R1-R6 → L3 → Tm5c → LoVP92 → DNg13
2,50,0.000005,R1-R6 → L3 → Tm5c → LoVP92 → DNg13
3,100,0.000003,R1-R6 → L3 → Tm5c → LoVP92 → VES200m → DNg13


## 8. Build every artefact the web app consumes

This writes `pathway.json` (types, routes, bottlenecks, removal effects,
sex mapping), `skeletons.json` (real EM-traced morphology, decimated) and
`cloud.json` (a background soma cloud), all in one shared coordinate frame.

In [13]:
from sight_to_action.build import build_all

paths = build_all()
for name, path in paths.items():
    print(f"{name:10s} {path}")

pathway    /Users/kiana/Desktop/Sight to Action/data/processed/pathway.json
skeletons  /Users/kiana/Desktop/Sight to Action/data/processed/skeletons.json
cloud      /Users/kiana/Desktop/Sight to Action/data/processed/cloud.json
explorer   /Users/kiana/Desktop/Sight to Action/data/processed/explorer.json
soma       /Users/kiana/Desktop/Sight to Action/data/processed/soma.json


## 9. Male vs female — the same analysis on the female connectome

Earlier versions of this project called their sex section a "comparison"
while only reading the dimorphism flags recorded in MaleCNS. That is
somebody else's result. Here the identical graph construction, relative
weight definition and route search are re-run on the **female** FlyWire
whole-brain connectome, so the comparison is actually computed.

FlyWire is brain-only (no ventral nerve cord), so descending neurons are
present but truncated at the neck.

In [14]:
from sight_to_action.female import (
    build_female_type_graph,
    load_female_annotations,
    load_female_connections,
    male_to_female_map,
    compare_edges,
    CONNECTIONS,
)

if CONNECTIONS.exists():
    female_ann = load_female_annotations()
    female_graph = build_female_type_graph(load_female_connections(), female_ann, min_weight=20)
    print(f"female graph: {female_graph.number_of_nodes():,} types, {female_graph.number_of_edges():,} edges")
else:
    female_graph = None
    print("FlyWire files not downloaded — see README. Skipping the female comparison.")

female graph: 8,772 types, 256,264 edges


### Which connections of the male pathway exist in the female brain?

In [15]:
if female_graph is not None:
    mapping = male_to_female_map(annotations, pathway_types)
    male_edges = [
        {"source": a, "target": b, "weight": graph.edges[a, b]["weight"],
         "relative_weight": graph.edges[a, b]["relative_weight"]}
        for a, b in graph.edges()
        if a in pathway_types and b in pathway_types
    ]
    cmp_df = pd.DataFrame(compare_edges(None, female_graph, male_edges, mapping))
    print(f"{cmp_df.found.sum()} / {len(cmp_df)} male pathway connections have a female counterpart")
    display_cols = ["male_source", "male_target", "male_relative_weight", "female_relative_weight", "found"]
    cmp_df[display_cols]

113 / 146 male pathway connections have a female counterpart


Every missing connection involves `LoVP92`, `VES200m` or `LC10c-1` — neurons
with no female counterpart. The connections that *do* match agree closely
across two independently reconstructed connectomes, which is reassuring:
`L1 → Tm3` is 20.1% in the male and 20.6% in the female, `L2 → Tm4` 22.5%
vs 21.8%.

### The strongest route in each sex, and the null test repeated

In [16]:
if female_graph is not None:
    from sight_to_action.nulls import best_scores_from

    female_routes = strongest_paths(female_graph, "R1-6", "DNg13", k=5, max_hops=5)
    print("male  :", " → ".join(routes[0].nodes))
    for i, r in enumerate(female_routes[:3], 1):
        print(f"female{i}:", " → ".join(r.nodes))

    dn_female = set(female_ann[female_ann["super_class"] == "descending"]["cell_type"].dropna())
    fwd_female = best_scores_from(female_graph, "R1-6", max_hops=5)
    ranked_female = sorted(
        ((t, s) for t, s in fwd_female.items() if t in dn_female and s > 0),
        key=lambda kv: kv[1],
        reverse=True,
    )
    pos = next((i for i, (t, _) in enumerate(ranked_female, 1) if t == "DNg13"), None)
    print(f"\nDNg13 rank among female descending neurons: {pos} / {len(ranked_female)}")
    print("top female DNs from the photoreceptors:", [t for t, _ in ranked_female[:6]])

male  :

 R1-R6 → L3 → Tm5c → LoVP92 → DNg13
female1: R1-6 → L1 → L5 → MTe01b → DNpe027 → DNg13
female2: R1-6 → L1 → Mi1 → Y3 → LTe42b → DNg13
female3: R1-6 → L2 → Tm1 → Y3 → LTe42b → DNg13

DNg13 rank among female descending neurons: 264 / 443
top female DNs from the photoreceptors: ['DNc01', 'DNc02', 'DNp11', 'DNp04', 'DNge102', 'DNg46']


**Conclusion.** The strongest male route runs through the male-specific
`LoVP92` and has no female equivalent; the female brain reaches DNg13 a
different way, while `R1-R6 → L1 → Mi1 → Y3 → LoVP90b → DNg13` is present in
both. And the null result replicates: DNg13 is below median in both sexes,
with nearly the same descending neurons at the top of both rankings, despite
the two connectomes being reconstructed by different groups. That
cross-dataset agreement is the strongest evidence here that the measure is
picking up biology rather than an artefact of one reconstruction.